# RAI2117307 Deep Learning Applications - Course Project
## Project 7: Wine Quality Score Prediction (Regression)

**Student:** Özgün Yolcu  
**Course:** RAI2117307 Deep Learning Applications  
**Institution:** Istanbul Medipol University, Vocational School - Robotics and AI  
**Repository:** [github.com/ozgunyolcu/DL-Wine-Quality](https://github.com/ozgunyolcu/DL-Wine-Quality)  
**Dataset:** Wine Quality (UCI Machine Learning Repository, ID: 186)  
**Task:** Regression (Target: Wine quality score 0–10)  
**Target Metrics:** Mean Absolute Error (MAE) and Coefficient of Determination ($R^2$)

---

### Pipeline Overview (Week 6 Template):
1. **Step 1:** Load the data and show examples
2. **Step 2:** Explore the data (EDA: distributions, summary statistics, correlation heatmap)
3. **Step 3:** Prepare the data (missing value analysis, train/validation/test split, feature scaling, PyTorch DataLoaders)
4. **Step 4:** Baseline Model (Simple 1-hidden-layer MLP)
5. **Step 5:** Improved Model (Deeper architecture, Dropout, Batch Normalization, LR Scheduler)
6. **Step 6:** Training curves (Train vs Validation Loss comparison)
7. **Step 7:** Test results (MAE, $R^2$, and Predicted vs True plot on isolated test set)
8. **Step 8:** Save the model (`torch.save`)
9. **Step 9:** Engineering conclusions and future directions

## Environment Setup & Library Imports

In [ ]:
# Install ucimlrepo if not already installed (required for loading the official dataset)
!pip install -q ucimlrepo

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

import torch
import torch.nn as nn
from torch.utils.data import TensorDataset, DataLoader

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import mean_absolute_error, r2_score
from ucimlrepo import fetch_ucirepo

# Set random seeds for strict reproducibility
SEED = 42
np.random.seed(SEED)
torch.manual_seed(SEED)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)

# Device configuration (GPU if available, CPU fallback)
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"Using compute device: {device}")

## 1. Load the Data and Show Examples
We fetch the Wine Quality dataset from the UCI Machine Learning Repository (ID: 186). The dataset contains physicochemical laboratory tests of red and white variants of the Portuguese *"Vinho Verde"* wine, with an associated sensory quality score between 0 and 10.

In [ ]:
# Fetch dataset using the official UCI ML Repository API
wine_dataset = fetch_ucirepo(id=186)

# Extract features (X) and target (y)
X = wine_dataset.data.features.copy()
y = wine_dataset.data.targets.copy()

# Target column name
target_col = y.columns[0]

print(f"Features shape: {X.shape} (instances, features)")
print(f"Target shape:   {y.shape}")
print("\nFirst 5 rows of input features:")
display(X.head())

In [ ]:
print("First 5 rows of target (quality scores):")
display(y.head())

## 2. Explore the Data (EDA)
Here we analyze:
- Total sample size and feature data types
- Summary statistics (mean, std, min, max, quartiles)
- Target variable distribution (verifying the clustering around 5, 6, and 7)
- Correlation between physicochemical features and wine quality

In [ ]:
# Dataset concise summary
print("=== Features Information ===")
X.info()

print("\n=== Descriptive Statistics of Features ===")
display(X.describe().T)

In [ ]:
# Plot 1: Target Variable (Quality Score) Distribution
plt.figure(figsize=(8, 4))
sns.countplot(x=y[target_col], palette="Blues_d")
plt.title("Distribution of Wine Quality Scores", fontsize=13, fontweight='bold')
plt.xlabel("Quality Score (0 to 10)", fontsize=11)
plt.ylabel("Count (Number of Samples)", fontsize=11)
plt.grid(axis='y', linestyle='--', alpha=0.7)

# Highlight the instructor's note: scores heavily concentrated at 5, 6, and 7
counts = y[target_col].value_counts()
for score, count in counts.items():
    plt.annotate(f"{count}", (score - y[target_col].min(), count + 25), ha='center', fontsize=10)

plt.tight_layout()
plt.show()

mean_val = y[target_col].mean()
median_val = y[target_col].median()
print(f"Target Mean:   {mean_val:.2f}")
print(f"Target Median: {median_val:.2f}")
print("Note: As observed, most samples have scores 5, 6, or 7. A trivial baseline predicting ~6 has low initial error.")

In [ ]:
# Plot 2: Correlation Heatmap between Features and Target
full_df = pd.concat([X, y], axis=1)
corr_matrix = full_df.corr()

plt.figure(figsize=(11, 8))
sns.heatmap(corr_matrix, annot=True, fmt=".2f", cmap="coolwarm", center=0, linewidths=0.5)
plt.title("Correlation Heatmap: Physicochemical Features vs Quality", fontsize=13, fontweight='bold')
plt.tight_layout()
plt.show()

print("\nCorrelation with Quality score:")
print(corr_matrix[target_col].sort_values(ascending=False))

## 3. Prepare the Data
In this phase we:
1. **Check for missing values:** Impute or verify cleanliness.
2. **Data Splitting:** Partition dataset into Training (70%), Validation (15%), and Test (15%).
3. **Feature Scaling:** Apply `StandardScaler`. To **prevent data leakage**, `fit` is computed **strictly on the training set**, and applied via `transform` to validation and test sets.
4. **PyTorch Pipeline:** Wrap tensors into `TensorDataset` and create mini-batch `DataLoader`s.

In [ ]:
# 3.1 Missing Value Check
missing_counts = X.isnull().sum()
print("Missing values per feature:")
print(missing_counts[missing_counts > 0] if missing_counts.sum() > 0 else "No missing values found in features.")

print(f"Target missing values: {y.isnull().sum().values[0]}")

In [ ]:
# 3.2 Train / Validation / Test Split (70% - 15% - 15%)
X_raw = X.values.astype(np.float32)
y_raw = y.values.astype(np.float32)  # shape: (N, 1)

# Step 1: Split into 70% Train, 30% Temp (Val + Test)
X_train, X_temp, y_train, y_temp = train_test_split(
    X_raw, y_raw, test_size=0.30, random_state=SEED
)

# Step 2: Split Temp evenly into 15% Validation and 15% Test
X_val, X_test, y_val, y_test = train_test_split(
    X_temp, y_temp, test_size=0.50, random_state=SEED
)

print(f"Training set samples:   {len(X_train)} ({len(X_train)/len(X_raw)*100:.1f}%)")
print(f"Validation set samples: {len(X_val)} ({len(X_val)/len(X_raw)*100:.1f}%)")
print(f"Test set samples:       {len(X_test)} ({len(X_test)/len(X_raw)*100:.1f}%)")

In [ ]:
# 3.3 Feature Scaling (StandardScaler)
# CRITICAL: Fit ONLY on training data to prevent information leakage!
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_val_scaled = scaler.transform(X_val)
X_test_scaled = scaler.transform(X_test)

print("Feature scaling successfully applied.")
print(f"Train mean across features: {X_train_scaled.mean(axis=0).round(2)}")
print(f"Train std across features:  {X_train_scaled.std(axis=0).round(2)}")

In [ ]:
# 3.4 Convert to PyTorch Tensors and DataLoaders
train_dataset = TensorDataset(torch.tensor(X_train_scaled, dtype=torch.float32), torch.tensor(y_train, dtype=torch.float32))
val_dataset   = TensorDataset(torch.tensor(X_val_scaled, dtype=torch.float32), torch.tensor(y_val, dtype=torch.float32))
test_dataset  = TensorDataset(torch.tensor(X_test_scaled, dtype=torch.float32), torch.tensor(y_test, dtype=torch.float32))

BATCH_SIZE = 64
train_loader = DataLoader(train_dataset, batch_size=BATCH_SIZE, shuffle=True)
val_loader   = DataLoader(val_dataset, batch_size=BATCH_SIZE, shuffle=False)
test_loader  = DataLoader(test_dataset, batch_size=BATCH_SIZE, shuffle=False)

print(f"DataLoaders created with batch size = {BATCH_SIZE}.")
print(f"Train batches: {len(train_loader)} | Val batches: {len(val_loader)} | Test batches: {len(test_loader)}")

## 4. Baseline Model (Week 8 Milestone Preview)
*A small neural network with one hidden layer, as specified in the course brief.*

In [ ]:
class BaselineWineNet(nn.Module):
    def __init__(self, input_dim=11, hidden_dim=32):
        super(BaselineWineNet, self).__init__()
        self.network = nn.Sequential(
            nn.Linear(input_dim, hidden_dim),
            nn.ReLU(),
            nn.Linear(hidden_dim, 1)
        )
        
    def forward(self, x):
        return self.network(x)

baseline_model = BaselineWineNet(input_dim=X_train.shape[1]).to(device)
print(baseline_model)

## 5. Improved Model (Week 9–13 Preview)
*An upgraded multi-layer deep network featuring Dropout regularization, Batch Normalization, and LeakyReLU activation to prevent overfitting on the quality distribution.*

In [ ]:
class ImprovedWineNet(nn.Module):
    def __init__(self, input_dim=11, hidden_dims=[64, 32, 16], dropout_rate=0.2):
        super(ImprovedWineNet, self).__init__()
        self.network = nn.Sequential(
            nn.Linear(input_dim, hidden_dims[0]),
            nn.BatchNorm1d(hidden_dims[0]),
            nn.LeakyReLU(0.1),
            nn.Dropout(dropout_rate),
            
            nn.Linear(hidden_dims[0], hidden_dims[1]),
            nn.BatchNorm1d(hidden_dims[1]),
            nn.LeakyReLU(0.1),
            nn.Dropout(dropout_rate),
            
            nn.Linear(hidden_dims[1], hidden_dims[2]),
            nn.ReLU(),
            
            nn.Linear(hidden_dims[2], 1)
        )
        
    def forward(self, x):
        return self.network(x)

improved_model = ImprovedWineNet(input_dim=X_train.shape[1]).to(device)
print(improved_model)